In [65]:
#here we will start using feature_engineered_dataset.csv to prepare ml model
import pandas as pd
import numpy as np
df=pd.read_csv("feature_engineered_dataset.csv")
df

,Platform,Year,Genre,Publisher,Global_Sales,Sales_Category
0,Wii,2006,Sports,Nintendo,82.74,Hit
1,NES,1985,Platform,Nintendo,40.24,Hit
2,Wii,2008,Racing,Nintendo,35.82,Hit
3,Wii,2009,Sports,Nintendo,33.00,Hit
4,GB,1996,Role-Playing,Nintendo,31.37,Hit
...,...,...,...,...,...,...
16286,GBA,2002,Platform,Kemco,0.01,Flop
16287,GC,2003,Shooter,Infogrames,0.01,Flop
16288,PS2,2008,Racing,Activision,0.01,Flop
16289,DS,2010,Puzzle,7G//AMES,0.01,Flop


In [66]:
df.dtypes

Platform              str
Year                int64
Genre                 str
Publisher             str
Global_Sales      float64
Sales_Category        str
dtype: object

In [67]:

top_publishers=df['Publisher'].value_counts().nlargest(15).index #Selected the top 15 most frequent publishers
df['Publisher_Grouped']=df['Publisher'].apply(lambda x:x if x in top_publishers else "Other")

In [68]:
#Regression Dataset
X_reg_input = df[['Platform', 'Year', 'Genre', 'Publisher_Grouped']]
y_reg_output=df['Global_Sales']
# Convert categorical string columns into one-hot encoded columns
X_reg_encoded=pd.get_dummies(X_reg_input,columns=['Platform','Genre','Publisher_Grouped'],drop_first=True)
X_reg_encoded


,Year,Platform_3DO,Platform_3DS,Platform_DC,Platform_DS,Platform_GB,Platform_GBA,Platform_GC,Platform_GEN,Platform_GG,...,Publisher_Grouped_Nintendo,Publisher_Grouped_Other,Publisher_Grouped_Sega,Publisher_Grouped_Sony Computer Entertainment,Publisher_Grouped_Square Enix,Publisher_Grouped_THQ,Publisher_Grouped_Take-Two Interactive,Publisher_Grouped_Tecmo Koei,Publisher_Grouped_Ubisoft,Publisher_Grouped_Warner Bros. Interactive Entertainment
0,2006,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1,1985,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,2008,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,2009,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,1996,False,False,False,False,True,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16286,2002,False,False,False,False,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,False
16287,2003,False,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,False,False,False,False
16288,2008,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
16289,2010,False,False,False,True,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


In [69]:
# Classification Dataset
# Since our input features (Platform, Year, Genre, Publisher) are identical to the regression setup, you can reuse X_reg_encoded:
y_class=df['Sales_Category']
X_class_encoded=X_reg_encoded.copy()
category_map={"Flop":0,"Average":1,"Hit":2}
y_class_encoded=df['Sales_Category'].map(category_map)


In [70]:
#Linear Regression Model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
# 1. Split the data (80% training, 20% testing)
X_train_reg,X_test_reg,y_train_reg,y_test_reg=train_test_split(X_reg_encoded,y_reg_output,test_size=0.2,random_state=42)

#Now lets Train The Regression Model
reg_model=LinearRegression()
reg_model.fit(X_train_reg,np.log1p(y_train_reg))

# Predict in log scale and convert back to actual units
y_pred_log = reg_model.predict(X_test_reg)
y_pred_reg = np.expm1(y_pred_log)

# Print actual target values
print("Actual values:\n", y_test_reg.values)

# Print predicted values (Negative values are gone!)
print("\nPredicted values:\n", y_pred_reg)

Actual values:
 [0.57 0.07 0.19 ... 0.09 0.13 0.19]

Predicted values:
 [0.29601482 0.10905401 0.08794643 ... 0.20668325 0.52275313 0.47157207]


In [71]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate metrics in original units
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R² Score: {r2:.4f}")

RMSE: 2.0121
MAE:  0.4829
R² Score: 0.0536


In [77]:
#Now lets train LogisticRegression Model(Predicting Sales_Category with Logistic Regression)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1.Lets first Split 
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_class_encoded, y_class_encoded, test_size=0.2, random_state=42
)

# 2. Create copies to avoid SettingWithCopyWarning
X_train_log = X_train_log.copy()
X_test_log = X_test_log.copy()

# 3. Fit scaler ONLY on X_train, then transform both train and test sets
scaler = StandardScaler()
X_train_log['Year'] = scaler.fit_transform(X_train_log[['Year']])
X_test_log['Year'] = scaler.transform(X_test_log[['Year']])

# 4. Train model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_log, y_train_log)

# 5. Predict & Evaluate
y_pred_log = log_model.predict(X_test_log)
print('Logistic Regression Results:')
print(f"Accuracy: {accuracy_score(y_test_log, y_pred_log):.4f}\n")

Logistic Regression Results:
Accuracy: 0.5707



In [81]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Split FIRST 
X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(
    X_class_encoded, y_class_encoded, test_size=0.2, random_state=42
)

# 2. Fit scaler strictly on training data, then transform both sets
scaler = StandardScaler()
X_train_knn_scaled = scaler.fit_transform(X_train_knn)
X_test_knn_scaled = scaler.transform(X_test_knn)

# 3. Train KNN model
knn_model = KNeighborsClassifier(n_neighbors=15)
knn_model.fit(X_train_knn_scaled, y_train_knn)

# 4. Predict & Evaluate
y_pred_knn = knn_model.predict(X_test_knn_scaled)

print("KNN Classifier Results:")
print(f"Accuracy: {accuracy_score(y_test_knn, y_pred_knn):.4f}")

KNN Classifier Results:
Accuracy: 0.5707
